# 08 - Model Training & 5-Fold Stratified Cross-Validation

## 1. Model Scope & Imbalance Mitigation
Per project requirements, only **Random Forest**, **XGBoost**, and **CatBoost** are evaluated.
All three models are configured with class weighting / scale positive weight to address the 9.6% minority class:
- **Random Forest:** `class_weight='balanced'`
- **XGBoost:** `scale_pos_weight=4.0`
- **CatBoost:** `auto_class_weights='Balanced'`

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import load_and_split_data
from src.models.train import get_candidate_models, cross_validate_model
from src.config import RAW_DATA_PATH

X_train, X_test, y_train, y_test = load_and_split_data(RAW_DATA_PATH)
candidate_models = get_candidate_models()

## 2. 5-Fold Stratified Cross-Validation Execution

In [ ]:
cv_results = {}
for name, model in candidate_models.items():
    print(f"--> Running 5-fold CV for {name}...")
    summary = cross_validate_model(name, model, X_train, y_train, n_splits=5)
    cv_results[name] = summary

## 3. Cross-Validation Results Comparison Table

In [ ]:
cv_summary_rows = []
for name, s in cv_results.items():
    cv_summary_rows.append({
        "Model": name,
        "CV Accuracy (Mean +/- Std)": f"{s['cv_accuracy_mean']:.4f} +/- {s['cv_accuracy_std']:.4f}",
        "CV Precision (Mean +/- Std)": f"{s['cv_precision_mean']:.4f} +/- {s['cv_precision_std']:.4f}",
        "CV Recall (Mean +/- Std)": f"{s['cv_recall_mean']:.4f} +/- {s['cv_recall_std']:.4f}",
        "CV F1 (Mean +/- Std)": f"{s['cv_f1_mean']:.4f} +/- {s['cv_f1_std']:.4f}",
        "CV PR-AUC (Mean +/- Std)": f"{s['cv_pr_auc_mean']:.4f} +/- {s['cv_pr_auc_std']:.4f}",
        "CV ROC-AUC (Mean +/- Std)": f"{s['cv_roc_auc_mean']:.4f} +/- {s['cv_roc_auc_std']:.4f}",
    })

cv_df = pd.DataFrame(cv_summary_rows)
display(cv_df)